# Ancestry Portability Analysis

## The Problem

Polygenic Risk Scores (PRS) are almost exclusively trained on European-ancestry GWAS data. When applied to individuals of other ancestries, prediction accuracy drops substantially. This is one of the most pressing equity issues in genomic medicine.

Two biological mechanisms drive this degradation:

1. **Allele frequency differences**: GWAS identifies SNPs that *tag* causal variants through correlation (LD). If the tag SNP has a very different frequency in another ancestry, the tagging relationship breaks down.

2. **LD structure differences**: The correlation structure between variants differs across ancestries. Effect sizes estimated in Europeans may be biased when applied to other populations because the underlying LD patterns are different.

We quantify this using **Fst (Fixation Index)** — a measure of genetic distance between populations:

$$F_{ST} = \frac{\bar{p}(1 - \bar{p}) - \frac{p_1(1-p_1) + p_2(1-p_2)}{2}}{\bar{p}(1-\bar{p})}$$

where $\bar{p} = (p_1 + p_2)/2$ is the mean allele frequency and $p_1$, $p_2$ are the frequencies in each population. $F_{ST} = 0$ means identical allele frequencies; $F_{ST} = 1$ means completely diverged.

---

## Setup

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Path resolution
ROOT = Path().resolve()
for candidate in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (candidate / 'genetics').exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')

from genetics.prs_pipeline import PRSModel
from genetics.ancestry_portability import (
    AncestryPortabilityAnalyzer,
    ANCESTRIES, ANCESTRY_ORDER, FREQ_MATRIX
)

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#0f0f0f',
    'text.color':       'white',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
})
print('Setup complete.')

## 1. Allele Frequency Differences Across Ancestries

First, let's visualize how AMD risk allele frequencies differ across populations. The variants with the largest frequency differences will contribute most to PRS portability degradation.

In [ ]:
rsids = list(PRSModel('AMD').variants['rsid'])
effects = PRSModel('AMD').variants['effect_size'].values

# Display frequency matrix
df_freqs = pd.DataFrame(
    FREQ_MATRIX,
    index=rsids,
    columns=ANCESTRY_ORDER
)
df_freqs['effect_size'] = effects
df_freqs.style.format('{:.3f}').background_gradient(cmap='Blues', subset=ANCESTRY_ORDER)

In [ ]:
# Visualize frequency differences for the 5 strongest variants
top5_idx = np.argsort(effects)[::-1][:5]
top5_rsids = [rsids[i] for i in top5_idx]
top5_effects = effects[top5_idx]

colors = [ANCESTRIES[a]['color'] for a in ANCESTRY_ORDER]
x = np.arange(len(top5_rsids))
width = 0.15

fig, ax = plt.subplots(figsize=(13, 5))
for i, (anc, color) in enumerate(zip(ANCESTRY_ORDER, colors)):
    freqs = FREQ_MATRIX[top5_idx, i]
    ax.bar(x + i * width, freqs, width, label=ANCESTRIES[anc]['label'], color=color, alpha=0.85)

ax.set_xticks(x + width * 2)
ax.set_xticklabels([f'{r}\n(β={e:.2f})' for r, e in zip(top5_rsids, top5_effects)], fontsize=9)
ax.set_ylabel('Risk Allele Frequency')
ax.set_title('Risk Allele Frequencies Across Ancestries — Top 5 AMD Variants', fontsize=12, fontweight='bold')
ax.legend(labelcolor='white', facecolor='#1a1a1a', edgecolor='#444', fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('#444')
ax.spines['left'].set_color('#444')
ax.set_ylim(0, 0.75)
plt.tight_layout()
plt.show()

print('Observation: rs1061170 (CFH) and rs10490924 (ARMS2) — the two strongest AMD loci —')
print('show the largest frequency differences between Europeans and Africans.')

## 2. Computing Fst Between Ancestries

Fst gives us a single number summarizing genetic distance between two populations across all variants.

In [ ]:
prs_model = PRSModel('AMD')
effect_sizes = prs_model.variants['effect_size'].values
analyzer = AncestryPortabilityAnalyzer(effect_sizes=effect_sizes, n_individuals=2000)

eur_freqs = FREQ_MATRIX[:, 0]

print('Fst values relative to European reference population:\n')
print(f'  {"Ancestry":<20} {"Fst":>8} {"Interpretation"}')
print(f'  {"-"*60}')
for i, anc in enumerate(ANCESTRY_ORDER):
    anc_freqs = FREQ_MATRIX[:, i]
    fst = analyzer.compute_fst(eur_freqs, anc_freqs)
    if fst < 0.01:
        interp = 'Training population (reference)'
    elif fst < 0.05:
        interp = 'Low genetic distance'
    elif fst < 0.10:
        interp = 'Moderate genetic distance'
    else:
        interp = 'High genetic distance'
    print(f'  {anc:<20} {fst:>8.4f}   {interp}')

## 3. PRS Portability Analysis

Now we run the full analysis. For each ancestry:
1. Simulate a population with ancestry-specific allele frequencies
2. Generate true disease labels using the liability threshold model
3. Compute PRS using European-trained effect sizes
4. Measure how well the European PRS predicts disease in each ancestry

This takes ~30 seconds to run.

In [ ]:
print('Running portability analysis (European-trained PRS)...')
result_eur = analyzer.run(n_bootstrap=200)

print('Running portability analysis (Multi-ancestry PRS)...')
result_multi = analyzer.run_multi_ancestry(n_bootstrap=200)

print('Done.')

In [ ]:
analyzer.print_summary(result_eur, result_multi)

## 4. Degradation Curve

The key result: prediction accuracy (R²) plotted as a function of Fst. This is the core visualization of the portability problem.

In [ ]:
analyzer.plot_degradation_curve(result_eur, result_multi)

## 5. Ancestry Comparison Bar Chart

In [ ]:
analyzer.plot_ancestry_comparison(result_eur, result_multi)

## 6. The Effect of Individual Variants on Portability

Not all variants contribute equally to portability degradation. Variants with large effect sizes AND large frequency differences across ancestries are the biggest culprits.

In [ ]:
# Compute per-variant contribution to portability degradation
eur_freqs = FREQ_MATRIX[:, 0]
afr_freqs = FREQ_MATRIX[:, 4]  # African — most diverged

freq_diff = np.abs(eur_freqs - afr_freqs)
contribution = freq_diff * effect_sizes  # frequency shift × effect size

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: frequency difference vs effect size scatter
ax = axes[0]
ax.set_facecolor('#0f0f0f')
scatter = ax.scatter(freq_diff, effect_sizes, c=contribution,
                     cmap='YlOrRd', s=80, edgecolors='white', linewidths=0.5, zorder=3)
for i, rsid in enumerate(rsids):
    if contribution[i] > np.percentile(contribution, 75):
        ax.annotate(rsid, (freq_diff[i], effect_sizes[i]),
                    xytext=(freq_diff[i]+0.003, effect_sizes[i]+0.01),
                    fontsize=7.5, color='white')
plt.colorbar(scatter, ax=ax, label='Portability impact score')
ax.set_xlabel('|Freq(EUR) - Freq(AFR)|')
ax.set_ylabel('Effect Size (β)')
ax.set_title('Per-Variant Portability Impact\n(EUR vs. African)', fontsize=11, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('#444'); ax.spines['left'].set_color('#444')

# Right: ranked bar chart of portability impact
ax = axes[1]
ax.set_facecolor('#0f0f0f')
sorted_idx = np.argsort(contribution)[::-1]
ax.barh(
    [rsids[i] for i in sorted_idx],
    contribution[sorted_idx],
    color=plt.cm.YlOrRd(contribution[sorted_idx] / contribution.max()),
    edgecolor='#444'
)
ax.set_xlabel('Portability Impact Score')
ax.set_title('Variants Ranked by Portability Impact', fontsize=11, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('#444'); ax.spines['left'].set_color('#444')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

print(f'Top 3 variants driving portability degradation (EUR → AFR):')
for i in sorted_idx[:3]:
    print(f'  {rsids[i]:<14}  Δfreq={freq_diff[i]:.3f}  β={effect_sizes[i]:.3f}  impact={contribution[i]:.4f}')

## Summary

| Finding | Result |
|---|---|
| R² in Europeans (training population) | Highest |
| R² in Africans (most diverged) | Substantially lower |
| Primary driver | CFH (rs1061170) and ARMS2 (rs10490924) — highest impact scores |
| Multi-ancestry re-weighting | Partially recovers accuracy across all non-European ancestries |

**Why this matters**: If AMD screening were based on PRS alone, individuals of African ancestry would be systematically under-identified as high-risk — a direct health equity consequence of the European bias in GWAS training data.

**What multi-ancestry GWAS does**: By jointly estimating effect sizes across diverse populations, LD mismatch is reduced, and the resulting PRS transfers more accurately. This is the approach advocated by the Global Biobank Meta-analysis Initiative and similar consortia.